# Full Hexagon Segmentation Pipeline

Reads `02_merged_data.parquet`, assigns H3 hexagons at a configurable resolution, and visualises trip-count and feature distributions per hexagon.

In [1]:
import pandas as pd
import polars as pl
import h3
import matplotlib.pyplot as plt
import numpy as np
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display
from shapely.geometry import shape
import geopandas 

## Load data

In [2]:
df = pl.read_parquet("../data/03_merged_data_with_h3.parquet").to_pandas()
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
# print("\nNumeric columns available for FEATURE:")
# print(sorted(df.select_dtypes(include='number').columns.tolist()))
# df.head(3)


Shape: 478,114 rows x 42 columns


In [3]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds',
       'trip_miles', 'pickup_census_tract', 'dropoff_census_tract',
       'pickup_community_area', 'dropoff_community_area', 'fare_usd',
       'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type',
       'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
       'different_days', 'temperature_2m', 'rain', 'precipitation',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'apparent_temperature', 'relative_humidity_2m', 'snow_depth',
       'snowfall', 'temperature_2m_max', 'temperature_2m_min', 'start_h3_r6',
       'end_h3_r6', 'start_h3_r7', 'end_h3_r7', 'start_h3_r8', 'end_h3_r8',
       'start_h3_r9', 'end_h3_r9'],
      dtype='str')

In [4]:
_H3_RESOLUTIONS = [6, 7, 8, 9]

def _build_geo(dataframe, h3_col):
    counts = dataframe.groupby(h3_col).count()['trip_id'].reset_index().rename(columns={'trip_id': 'trip_count'})
    counts.columns = ["h3_cell", "trip_count"]
    counts['geometry'] = counts.apply(lambda x: shape(h3.cells_to_geo([x["h3_cell"]])), axis=1)
    return geopandas.GeoDataFrame(counts, geometry=counts['geometry'], crs='EPSG:4326')

_starts_geo = {res: _build_geo(df, f"start_h3_r{res}") for res in _H3_RESOLUTIONS}
_ends_geo   = {res: _build_geo(df, f"end_h3_r{res}")   for res in _H3_RESOLUTIONS}

# # Keep originals pointing at resolution 8 for backward compatibility
# start_counts = _starts_geo[8].copy()
# end_counts   = _ends_geo[8].copy()
# starts_geo   = _starts_geo[8]
# ends_geo     = _ends_geo[8]


In [5]:
_starts_geo

{6:             h3_cell  trip_count  \
 0   862664197ffffff         221   
 1   8626641b7ffffff          15   
 2   862664527ffffff        1048   
 3   86266452fffffff          46   
 4   862664567ffffff         918   
 5   86266456fffffff         106   
 6   862664577ffffff           5   
 7   862664c17ffffff       43649   
 8   862664c1fffffff      334147   
 9   862664c87ffffff         280   
 10  862664c8fffffff         220   
 11  862664ca7ffffff        3659   
 12  862664cafffffff       36546   
 13  862664cb7ffffff         454   
 14  862664cc7ffffff        5793   
 15  862664ccfffffff        1397   
 16  862664cd7ffffff         339   
 17  862664cdfffffff         601   
 18  862664ce7ffffff        1668   
 19  862664cefffffff         804   
 20  862664cf7ffffff       11248   
 21  862664d8fffffff       17318   
 22  862664d9fffffff         706   
 23  862759347ffffff       15649   
 24  86275934fffffff          36   
 25  86275936fffffff         383   
 
                       

## Map: trip count per hexagon

Toggle between **Pickup hexagons** and **Dropoff hexagons** using the layer control (top-right). Colour intensity encodes number of trips; hover a hexagon for the exact count.

In [6]:
all_lats = pd.concat([df["pickup_lat"], df["dropoff_lat"]], ignore_index=True).dropna()
all_lons = pd.concat([df["pickup_lon"], df["dropoff_lon"]], ignore_index=True).dropna()
map_center = [all_lats.mean(), all_lons.mean()]


def _add_hex_layer(map_obj, counts_df, layer_name, show, colormap):
    layer = folium.FeatureGroup(name=layer_name, show=show)
    for _, row in counts_df.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = colormap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.65,
            tooltip=folium.Tooltip(f"{layer_name}: {row['trip_count']:,} trips"),
        ).add_to(layer)
    layer.add_to(map_obj)


_trips_res_dropdown = widgets.Dropdown(
    options=[(f"Resolution {r}", r) for r in _H3_RESOLUTIONS],
    value=8,
    description="Resolution:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_trips_out = widgets.Output()


def _build_trips_map(res):
    sg = _starts_geo[res]
    eg = _ends_geo[res]
    max_count = int(max(
        sg["trip_count"].max() if not sg.empty else 0,
        eg["trip_count"].max() if not eg.empty else 0,
    ))
    colormap = cm.linear.YlOrRd_09.scale(0, max_count)
    colormap.caption = "Trips per hexagon"
    m = folium.Map(location=map_center, zoom_start=11, tiles="CartoDB positron")
    _add_hex_layer(m, sg, "Pickup hexagons",  True, colormap)
    _add_hex_layer(m, eg, "Dropoff hexagons", True, colormap)
    colormap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_trips_res_change(change):
    with _trips_out:
        _trips_out.clear_output(wait=True)
        display(_build_trips_map(change["new"]))


_trips_res_dropdown.observe(_on_trips_res_change, names="value")

with _trips_out:
    display(_build_trips_map(8))

display(widgets.VBox([_trips_res_dropdown, _trips_out]))


## Map: feature value per hexagon (interactive)

Use the **dropdown** to pick any numeric feature. The map re-renders immediately, colouring each pickup hexagon by its **mean value**. Hover a hexagon to see the exact value and trip count.

In [7]:
SELECTABLE_FEATURES = [
    "apparent_temperature", "extras_usd",
    "fare_usd",
    "precipitation", "rain", "relative_humidity_2m", "snow_depth",
    "temperature_2m", "temperature_2m_max", "temperature_2m_min",
    "tips_usd", "tolls_usd", "trip_miles", "trip_seconds", "trip_total_usd",
    "wind_direction_10m", "wind_gusts_10m", "wind_speed_10m",
]
SELECTABLE_FEATURES = [f for f in SELECTABLE_FEATURES if f in df.columns]

FEATURE       = "fare_usd"  # any numeric column (e.g. trip_miles, tips_usd, temperature_2m)

_all_lats = pd.concat([df["pickup_lat"], df["dropoff_lat"]], ignore_index=True).dropna()
_all_lons = pd.concat([df["pickup_lon"], df["dropoff_lon"]], ignore_index=True).dropna()
_map_center = [_all_lats.mean(), _all_lons.mean()]


def _build_feature_map(feature, res):
    h3_col = f"start_h3_r{res}"
    agg = (
        df.dropna(subset=[h3_col, feature])
        .groupby(h3_col)[feature]
        .agg(["mean", "count"])
        .reset_index()
    )
    agg.columns = ["h3_cell", "mean_val", "trip_count"]

    feat_min, feat_max = agg["mean_val"].min(), agg["mean_val"].max()
    colormap = cm.linear.PuBuGn_09.scale(feat_min, feat_max)
    colormap.caption = f"Mean {feature} per hexagon"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")
    layer = folium.FeatureGroup(name=f"Mean {feature}", show=True)

    for _, row in agg.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = colormap(row["mean_val"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"Mean {feature}: {row['mean_val']:.2f}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    colormap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


_initial = FEATURE if FEATURE in SELECTABLE_FEATURES else SELECTABLE_FEATURES[0]

_dropdown = widgets.Dropdown(
    options=SELECTABLE_FEATURES,
    value=_initial,
    description="Feature:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
_feat_res_dropdown = widgets.Dropdown(
    options=[(f"Resolution {r}", r) for r in _H3_RESOLUTIONS],
    value=8,
    description="Resolution:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_out = widgets.Output()


def _on_feat_change(change):
    with _out:
        _out.clear_output(wait=True)
        display(_build_feature_map(_dropdown.value, _feat_res_dropdown.value))


_dropdown.observe(_on_feat_change, names="value")
_feat_res_dropdown.observe(_on_feat_change, names="value")

with _out:
    display(_build_feature_map(_initial, 8))

display(widgets.VBox([widgets.HBox([_dropdown, _feat_res_dropdown]), _out]))


## Map: trips per hexagon by day of week

Select a **day of the week** (or "All days") and toggle between **Pickup** and **Dropoff** hexagons. Colour intensity encodes number of trips for that day. `different_days` encodes 0 = Monday … 6 = Sunday.

In [8]:
_DOW_NAMES = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday",
}

_dow_day_dropdown = widgets.Dropdown(
    options=[("All days", -1)] + [(name, i) for i, name in _DOW_NAMES.items()],
    value=-1,
    description="Day:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_dow_hex_dropdown = widgets.Dropdown(
    options=[("Pickup hexagons", "start"), ("Dropoff hexagons", "end")],
    value="start",
    description="Trip end:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_dow_res_dropdown = widgets.Dropdown(
    options=[(f"Resolution {r}", r) for r in _H3_RESOLUTIONS],
    value=8,
    description="Resolution:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_dow_out = widgets.Output()


def _build_dow_map(day_val, trip_type, res):
    h3_col = f"{trip_type}_h3_r{res}"
    subset = df if day_val == -1 else df[df["different_days"] == day_val]

    counts = (
        subset.dropna(subset=[h3_col])[h3_col]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["h3_cell", "trip_count"]

    day_label = "All days" if day_val == -1 else _DOW_NAMES[day_val]
    hex_label = "Pickup" if trip_type == "start" else "Dropoff"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")

    if counts.empty:
        return m

    cmap = cm.linear.YlOrRd_09.scale(0, int(counts["trip_count"].max()))
    cmap.caption = f"{hex_label} trips per hexagon — {day_label}"

    layer = folium.FeatureGroup(name=f"{hex_label} · {day_label}", show=True)
    for _, row in counts.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = cmap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"{hex_label} · {day_label}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_dow_change(change):
    with _dow_out:
        _dow_out.clear_output(wait=True)
        display(_build_dow_map(_dow_day_dropdown.value, _dow_hex_dropdown.value, _dow_res_dropdown.value))


_dow_day_dropdown.observe(_on_dow_change, names="value")
_dow_hex_dropdown.observe(_on_dow_change, names="value")
_dow_res_dropdown.observe(_on_dow_change, names="value")

with _dow_out:
    display(_build_dow_map(-1, "start", 8))

display(widgets.VBox([
    widgets.HBox([_dow_day_dropdown, _dow_hex_dropdown, _dow_res_dropdown]),
    _dow_out,
]))


## Map: trips per hexagon by hour of day

Select an **hour** (0–23, or "All hours") and toggle between **Pickup** and **Dropoff** hexagons. Hour is derived from `trip_start`.

In [9]:
_trip_hour = df["trip_start"].dt.hour  # computed once, reused on every map render

_hour_day_dropdown = widgets.Dropdown(
    options=[("All hours", -1)] + [(f"{h:02d}:00", h) for h in range(24)],
    value=-1,
    description="Hour:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_hour_hex_dropdown = widgets.Dropdown(
    options=[("Pickup hexagons", "start"), ("Dropoff hexagons", "end")],
    value="start",
    description="Trip end:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_hour_res_dropdown = widgets.Dropdown(
    options=[(f"Resolution {r}", r) for r in _H3_RESOLUTIONS],
    value=8,
    description="Resolution:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_hour_out = widgets.Output()


def _build_hour_map(hour_val, trip_type, res):
    h3_col = f"{trip_type}_h3_r{res}"
    mask = slice(None) if hour_val == -1 else (_trip_hour == hour_val)
    subset = df if hour_val == -1 else df[mask]

    counts = (
        subset.dropna(subset=[h3_col])[h3_col]
        .value_counts()
        .reset_index()
    )
    counts.columns = ["h3_cell", "trip_count"]

    hour_label = "All hours" if hour_val == -1 else f"{hour_val:02d}:00"
    hex_label  = "Pickup" if trip_type == "start" else "Dropoff"

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")

    if counts.empty:
        return m

    cmap = cm.linear.BuPu_09.scale(0, int(counts["trip_count"].max()))
    cmap.caption = f"{hex_label} trips per hexagon — {hour_label}"

    layer = folium.FeatureGroup(name=f"{hex_label} · {hour_label}", show=True)
    for _, row in counts.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = cmap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"{hex_label} · {hour_label}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)

    layer.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_hour_change(change):
    with _hour_out:
        _hour_out.clear_output(wait=True)
        display(_build_hour_map(_hour_day_dropdown.value, _hour_hex_dropdown.value, _hour_res_dropdown.value))


_hour_day_dropdown.observe(_on_hour_change, names="value")
_hour_hex_dropdown.observe(_on_hour_change, names="value")
_hour_res_dropdown.observe(_on_hour_change, names="value")

with _hour_out:
    display(_build_hour_map(-1, "start", 8))

display(widgets.VBox([
    widgets.HBox([_hour_day_dropdown, _hour_hex_dropdown, _hour_res_dropdown]),
    _hour_out,
]))


## Plots: trip count per hexagon by 8-hour block

Three side-by-side choropleth maps — one per 8-hour block (00–07, 08–15, 16–23). Toggle between **Pickup** and **Dropoff** trips.

In [10]:
from IPython.display import HTML

df['start_hour'] = pd.to_datetime(df['trip_start']).dt.hour
df['end_hour']   = pd.to_datetime(df['trip_end']).dt.hour
df['start_eight_hour'] = df['start_hour'] // 8
df['end_eight_hour']   = df['end_hour']   // 8

_BLOCK_LABELS = {0: "00:00–07:59", 1: "08:00–15:59", 2: "16:00–23:59"}
_BLOCK_CMAPS  = [cm.linear.Blues_09, cm.linear.Greens_09, cm.linear.Oranges_09]

_eight_hex_dropdown = widgets.Dropdown(
    options=[("Pickup hexagons", "start"), ("Dropoff hexagons", "end")],
    value="start",
    description="Trip type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_eight_block_dropdown = widgets.Dropdown(
    options=[(label, i) for i, label in _BLOCK_LABELS.items()],
    value=0,
    description="Block:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_eight_res_dropdown = widgets.Dropdown(
    options=[(f"Resolution {r}", r) for r in _H3_RESOLUTIONS],
    value=8,
    description="Resolution:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)
_eight_out = widgets.Output()


def _build_eight_hour_map(trip_type, block, res):
    h3_col    = f"{trip_type}_h3_r{res}"
    block_col = "start_eight_hour" if trip_type == "start" else "end_eight_hour"
    label     = "Pickup" if trip_type == "start" else "Dropoff"

    subset = df[df[block_col] == block].dropna(subset=[h3_col])
    counts = subset[h3_col].value_counts().reset_index()
    counts.columns = ["h3_cell", "trip_count"]

    m = folium.Map(location=_map_center, zoom_start=11, tiles="CartoDB positron")

    if counts.empty:
        return m

    cmap = _BLOCK_CMAPS[block].scale(0, int(counts["trip_count"].max()))
    cmap.caption = f"{label} trips — {_BLOCK_LABELS[block]}"

    layer = folium.FeatureGroup(name=f"{label} · {_BLOCK_LABELS[block]}", show=True)
    for _, row in counts.iterrows():
        boundary = h3.cell_to_boundary(row["h3_cell"])
        color = cmap(row["trip_count"])
        folium.Polygon(
            locations=[[lat, lon] for lat, lon in boundary],
            color=color, weight=1, fill=True,
            fill_color=color, fill_opacity=0.7,
            tooltip=folium.Tooltip(
                f"{_BLOCK_LABELS[block]}<br>Trips: {row['trip_count']:,}"
            ),
        ).add_to(layer)
    layer.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m


def _on_eight_change(change):
    with _eight_out:
        _eight_out.clear_output(wait=True)
        display(_build_eight_hour_map(_eight_hex_dropdown.value, _eight_block_dropdown.value, _eight_res_dropdown.value))


_eight_hex_dropdown.observe(_on_eight_change, names="value")
_eight_block_dropdown.observe(_on_eight_change, names="value")
_eight_res_dropdown.observe(_on_eight_change, names="value")

with _eight_out:
    display(_build_eight_hour_map("start", 0, 8))

display(widgets.VBox([
    widgets.HBox([_eight_hex_dropdown, _eight_block_dropdown, _eight_res_dropdown]),
    _eight_out,
]))
